# GigBid: one agent, a supervisor, or the hand-built graph? (R&D question C) on a free Kaggle GPU

About 45 minutes. The same 30 dev jobs (10 per persona) go through three architectures with the same local model and the same MCP tools:
a single ReAct agent, a `langgraph-supervisor` team, and the hand-built bid graph. Each must say bid or skip, and whether the job is a scam.
The hypothesis is written at the top of `evals/runners/arch_study.py`.

It writes `evals/results/arch_study.json` after every architecture and prints it at the end with its SHA-256.

Before you run it, set **Accelerator: GPU T4 x2** and **Internet: On**.


In [ ]:
# 1. Get the code
REPO = "https://github.com/aimennnnn/gigbid.git"
!rm -rf /kaggle/working/gigbid
!git clone --depth 1 $REPO /kaggle/working/gigbid
%cd /kaggle/working/gigbid
!git log --oneline -1

In [ ]:
# 2. Install the project (uv reads pyproject.toml and uv.lock, the same versions as on the laptop)
!pip install -q uv
!uv sync --frozen

In [ ]:
# 3. Install Ollama, start it in the background and download the model (about 14 GB)
import os
import subprocess
import time
!apt-get install -y -qq pciutils lshw zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
# a bigger context window than Ollama's default, so long prompts (job + evidence + schema) are not cut off
server = subprocess.Popen(["ollama", "serve"], env=os.environ | {"OLLAMA_CONTEXT_LENGTH": "16384"},
                          stdout=open("/kaggle/working/ollama.log", "w"), stderr=subprocess.STDOUT)
for attempt in range(1, 11):          # the model registry sometimes answers "503": wait and try again
    time.sleep(15)
    pull = subprocess.run(["ollama", "pull", "gpt-oss:20b"], capture_output=True, text=True)
    if pull.returncode == 0:
        break
    print(f"download attempt {attempt} failed: {pull.stderr.strip()[-200:]}")
models = subprocess.run(["ollama", "list"], capture_output=True, text=True).stdout
print(models)
assert "gpt-oss:20b" in models, "the model was not downloaded, so the LLM steps cannot run"

In [ ]:
# 4. Point GigBid at the local model. These names are read by agent/llm.py (see agent/config.yaml, provider "ollama").
os.environ["OLLAMA_URL"] = "http://localhost:11434/v1"
os.environ["GIGBID_PROVIDER"] = "ollama"

# the hardware goes into the results folder, because the report must state it for local models
!mkdir -p evals/results
!nvidia-smi --query-gpu=name,memory.total --format=csv | tee evals/results/kaggle_hardware.txt
!nproc | xargs echo "CPU cores:" | tee -a evals/results/kaggle_hardware.txt

In [ ]:
# 5. The study: 30 jobs x 3 architectures. The baselines run first; the graph's numbers on 60 jobs already exist.
!uv run python -m evals.runners.arch_study --limit 10


In [ ]:
# 6. Print the result file and its hash.
import hashlib
import pathlib
raw = pathlib.Path("evals/results/arch_study.json").read_bytes()
print("SHA256", hashlib.sha256(raw).hexdigest(), "BYTES", len(raw))
print("-----BEGIN arch_study.json-----")
print(raw.decode("utf-8"))
print("-----END arch_study.json-----")
